# ROZPORZĄDZENIE MINISTRA INFRASTRUKTURY 
z dnia 8 listopada 2024 r. w sprawie rejestracji i oznaczania pojazdów, wymagań dla tablic rejestracyjnych oraz wzorów innych dokumentów związanych z rejestracją pojazdów (source: https://www.dziennikustaw.gov.pl/D2024000170901.pdf "D2024000170901.pdf")

In [0]:
%pip install lxml
%pip install pdfplumber

In [0]:
import os
from pyspark.sql.functions import to_json, col
import json
from io import StringIO
import pandas as pd
import lxml
from pyspark.sql.functions import col, concat_ws, initcap
from delta.tables import DeltaTable
import pdfplumber

In [0]:
FILE_PATH = "/Volumes/workspace/default/undercover_cars/D2024000170901_plates.pdf"
df = spark.read.format("binaryFile").load(FILE_PATH)
display(df.select("path", "modificationTime", "length"))

## 2. Parse or analyze the file with AI functions Prerequisite: AI functions require a workspace in a supported region.

In [0]:

# FILE_PATH is assumed to be set already
file_type = os.path.splitext(FILE_PATH)[1].lower()
# 1) Tabular / structured files
if file_type in [".csv", ".parquet", ".json"]:
    print(f"Detected structured file: {file_type}")
    if file_type == ".csv":
        df = spark.read.option("header", True).csv(FILE_PATH)
    elif file_type == ".parquet":
        df = spark.read.parquet(FILE_PATH)
    else:  # .json
        df = spark.read.json(FILE_PATH)
    display(df)
    df.printSchema()
# 2) Documents: parse with ai_parse_document
elif file_type in [".pdf", ".doc", ".docx", ".ppt", ".pptx"]:
    print(f"Detected document file: {file_type}")
    print("Parsing content using ai_parse_document...")
    # Step 1 - Read the file as binary
    df = spark.read.format("binaryFile").load(FILE_PATH)
    # Step 2 - Parse the document with AI
    parsed = df.selectExpr(
        "path AS file_path",
        "ai_parse_document(content, map('version', '2.0')) AS parsed_content"
    )
    # Step 3 - Display safely (ai_parse_document returns a complex type)
    parsed_for_display = parsed.select(
        col("file_path"),
        to_json(col("parsed_content")).alias("parsed_content_json")
    )
    display(parsed_for_display)
# 3) Images: analyze with ai_query
elif file_type in [".jpg", ".jpeg", ".png"]:
    print(f"Detected image file: {file_type}")
    print("Analyzing content using ai_query...")
    # Step 1 - Read the file as binary
    df = spark.read.format("binaryFile").load(FILE_PATH)
    # Step 2 - Analyze the image with AI
    analyzed = df.selectExpr(
        "path AS file_path",
        "ai_query('databricks-llama-4-maverick', 'Describe this image in one sentence:', files => content) AS description"
    ).filter("length < 5000000")
    display(analyzed)
# 4) Fallback
else:
    print(f"Unsupported file type: {file_type}")
    print("You can still inspect file metadata:")
    display(
        spark.read.format("binaryFile")
        .load(FILE_PATH)
        .select("path", "modificationTime", "length")
    )

In [0]:
# local/FUSE path (Volumes or /dbfs/...), not dbfs:/
pdf_path = "/Volumes/workspace/default/undercover_cars/D2024000170901_plates.pdf"

# pdf_path = "/Volumes/<catalog>/<schema>/<volume>/D2024000170901_plates.pdf"
SKIP_FIRST_PAGE_ROWS = 2

schema = "`Lp` STRING, `Województwo` STRING, `Powiat` STRING, `Wyróżnik Województwa` STRING, `Wyróżnik Powiatu` STRING, `Wyróżnik Tymczasowe` STRING"

# 1) raw rows from all pages; skip the first 2 rows of page 1 only
raw = []
with pdfplumber.open(pdf_path) as pdf:
    for page_no, page in enumerate(pdf.pages):
        for tbl in page.extract_tables():
            raw.extend(tbl[SKIP_FIRST_PAGE_ROWS:] if page_no == 0 else tbl)

clean = [[(c or "").replace("\n", " ").strip() for c in r] for r in raw]
clean = [r for r in clean if r[:6] != ["1", "2", "3", "4", "5", "6"]]  # column-number row

# 2) glue wrapped cells (e.g. "KUJAWSKO-" + "-POMORSKIE", "D0–D9," + "V0–V9") to the previous row
merged = []
for r in clean:
    if merged and not r[0] and not r[2] and not r[3] and not r[4]:
        prev = merged[-1]
        for i in (1, 5):
            if r[i]:
                prev[i] = prev[i] + r[i].lstrip("-") if prev[i].endswith("-") else f"{prev[i]} {r[i]}".strip()
    else:
        merged.append(r)

# 3) fill down Lp / Województwo / Wyróżnik Tymczasowe from each voivodeship row
records, ctx = [], ["", "", ""]
for lp, woj, powiat, w_woj, w_pow, w_tmp in merged:
    if lp:
        ctx = [lp, woj, w_tmp]
    records.append((ctx[0], ctx[1], powiat or None, w_woj, w_pow or None, ctx[2]))

df = spark.createDataFrame(records, schema=schema) \
    .filter("Powiat IS NOT NULL").drop("Lp", "Wyróżnik Tymczasowe")


display(df)

In [0]:
# Expand the combined table: within each Województwo, take every value of
# "Wyróżnik Województwa" (comma-separated, e.g. "D, V") and cross it with
# every value of "Wyróżnik Powiatu" (also comma-separated) to produce all
# plate combinations in a new "plate_prefix" column.
def split_values(v):
    """Split a possibly comma-separated cell into a clean list of values."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    return [part.strip() for part in str(v).split(",") if part.strip()]


# df arrives as a Spark DataFrame -> switch to pandas for the row-wise expansion below (skip when it is already pandas from a previous run)
if hasattr(df, "toPandas"):
    df = df.toPandas()

# The voivodeship indicator may only be filled on some rows of a group;
# propagate the first non-null value to every row of that Województwo so the
# cartesian product is scoped to the Województwo window.
woj_wyrzoznik_map = (
    df.dropna(subset=["Wyróżnik Województwa"])
    .groupby("Województwo")["Wyróżnik Województwa"]
    .first()
)
df["Wyróżnik Województwa"] = df["Województwo"].map(woj_wyrzoznik_map)


# Cartesian product: each województwa value x each powiat value, e.g.
# "D, V" x "C, K" -> "D C", "D K", "V C", "V K"
def plate_combinations(row):
    woj_values = split_values(row["Wyróżnik Województwa"])
    pow_values = split_values(row["Wyróżnik Powiatu"])
    return [f"{w}{p}" for w in woj_values for p in pow_values]


plates = (
    df.assign(plate_prefix=df.apply(plate_combinations, axis=1))
    .explode("plate_prefix")
    .dropna(subset=["plate_prefix"])
    .reset_index(drop=True)
)

# Show, per Województwo, how many plate combinations were produced
# print(
#     plates.groupby("Województwo")["plate_prefix"]
#     .nunique()
#     .rename("unique_plates")
# )
# print(f"Total plate combinations: {len(plates)}")

# Publish as a Spark DataFrame with df's columns plus the new plate_prefix column
plates_columns = [c for c in df.columns] + ["plate_prefix"]
plates_schema = ", ".join(f"`{c}` STRING" for c in plates_columns)
plates_df = spark.createDataFrame(plates[plates_columns], plates_schema) \
    .withColumn("Województwo", initcap(col("Województwo"))) \
    .withColumn("Entity", concat_ws(", pow. ", col("Województwo"), col("Powiat"))) \
    .drop("Lp", "Wyróżnik Tymczasowe", "Wyróżnik Powiatu", "Wyróżnik Województwa", "Powiat", "Województwo") 
display(plates_df)

In [0]:
custom_pl_plates_prefixes_entities = {
    'HA': 'Służba: Centralne Biuro Antykorupcyjne',
    'HB': 'Służba: Służba Ochrony Państwa',
    'HC': 'Służba: Służba Celno-Skarbowa (dawniej Służba Celna)',
    'HK': 'Służba: Agencja Bezpieczeństwa Wewnętrznego i Agencja Wywiadu',
    'HM': 'Służba: Służba Kontrwywiadu Wojskowego i Służba Wywiadu Wojskowego',
    'HP': 'Służba: Policja',
    'HS': 'Służba: Kontrola Skarbowa (dawna)',

    'UA': 'Wojsko (Siły Zbrojne RP): samochody osobowe/terenowe/specjalne na podwoziu osobowym; czołgi (zasób II)',
    'UB': 'Wojsko (Siły Zbrojne RP): transportery opancerzone',
    'UC': 'Wojsko (Siły Zbrojne RP): lekkie samochody ciężarowe i dostawcze; bojowe wozy piechoty (zasób II)',
    'UD': 'Wojsko (Siły Zbrojne RP): autobusy',
    'UE': 'Wojsko (Siły Zbrojne RP): samochody ciężarowe do transportu; pojazdy wsparcia technicznego (zasób II)',
    'UF': 'Wojsko (Siły Zbrojne RP): wyrzutnie rakietowe (zasób II)',
    'UG': 'Wojsko (Siły Zbrojne RP): pojazdy specjalne na podwoziu ciężarowym; mosty czołgowe (zasób II)',
    'UH': 'Wojsko (Siły Zbrojne RP): działa samobieżne (zasób II)',
    'UI': 'Wojsko (Siły Zbrojne RP): przyczepy transportowe; pojazdy inżynieryjne (zasób II)',
    'UJ': 'Wojsko (Siły Zbrojne RP): przyczepy specjalne; pojazdy ewakuacji medycznej (zasób II)',
    'UK': 'Wojsko (Siły Zbrojne RP): motocykle',
    'UL': 'Wojsko (Siły Zbrojne RP): zmobilizowane pojazdy cywilne; pojazdy dowodzenia (zasób II)',
    'UN': 'Wojsko (Siły Zbrojne RP): samobieżne działa przeciwlotnicze (zasób II)',
    'UR': 'Wojsko (Siły Zbrojne RP): pojazdy rozpoznawcze (zasób II)',

    'HCA': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Olsztynie',
    'HCB': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Białymstoku',
    'HCC': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Lublinie',
    'HCD': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Rzeszowie',
    'HCE': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Krakowie',
    'HCF': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Katowicach',
    'HCG': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej we Wrocławiu',
    'HCH': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Zielonej Górze',
    'HCJ': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Szczecinie',
    'HCK': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Gdańsku',
    'HCL': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Warszawie',
    'HCM': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Bydgoszczy',
    'HCN': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Łodzi',
    'HCO': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Poznaniu',
    'HCP': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Opolu',
    'HCR': 'Służba Celno-Skarbowa: Izba Administracji Skarbowej w Kielcach',
    'HPA': 'Jednostka Policji: Komenda Główna Policji',
    'HPB': 'Jednostka Policji: KWP Wrocław',
    'HPC': 'Jednostka Policji: KWP Bydgoszcz',
    'HPD': 'Jednostka Policji: KWP Lublin',
    'HPE': 'Jednostka Policji: KWP Gorzów Wielkopolski',
    'HPF': 'Jednostka Policji: KWP Łódź',
    'HPG': 'Jednostka Policji: KWP Kraków',
    'HPH': 'Jednostka Policji: KWP z siedzibą w Radomiu',
    'HPJ': 'Jednostka Policji: KWP Opole',
    'HPK': 'Jednostka Policji: KWP Rzeszów',
    'HPL': 'Jednostka Policji: szkoły Policji: Akademia Policji (Szczytno), Szkoły Policji (Piła, Słupsk, Katowice), Centrum Szkolenia Policji (Legionowo)',
    'HPM': 'Jednostka Policji: KWP Białystok',
    'HPN': 'Jednostka Policji: KWP Gdańsk',
    'HPP': 'Jednostka Policji: KWP Katowice',
    'HPS': 'Jednostka Policji: KWP Kielce',
    'HPT': 'Jednostka Policji: KWP Olsztyn',
    'HPU': 'Jednostka Policji: KWP Poznań',
    'HPW': 'Jednostka Policji: KWP Szczecin',
    'HPZ': 'Jednostka Policji: Komenda Stołeczna Policji (Warszawa)',
}

In [0]:
# Turn the {prefix: entity} mapping into a Spark DataFrame with the same shape as plates_df (plate_prefix, Entity)
custom_df = spark.createDataFrame(
    custom_pl_plates_prefixes_entities.items(), "plate_prefix STRING, Entity STRING"
)

# Union both sources, aligning by column name and ordering columns explicitly
all_plates_df = (
    plates_df.select("plate_prefix", "Entity")
    .unionByName(custom_df.select("plate_prefix", "Entity"))
    .orderBy(col("plate_prefix"))
)
display(all_plates_df)
print(f"plates_df rows: {plates_df.count()}, custom rows: {custom_df.count()}, total: {all_plates_df.count()}")

In [0]:
# Target Delta table for the plate dimension
TARGET_CATALOG = "workspace"
TARGET_SCHEMA = "undercover_cars"
TARGET_TABLENAME = "dim_pl_registration_plates_pattern"
TARGET_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLENAME}"

spark.sql(f"USE CATALOG {TARGET_CATALOG}")

# Make sure the target schema exists before writing to it
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")



# Source of (new) values: plate_prefix + Entity from the union above
source_df = all_plates_df.select("plate_prefix", "Entity")

if not spark.catalog.tableExists(TARGET_TABLE):
    # First run: create the table directly from the source data
    source_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
    print(f"Table {TARGET_TABLE} did not exist - created with {source_df.count()} rows.")
else:
    # Subsequent runs: upsert on plate_prefix - insert new prefixes,
    # refresh Entity when it changed for an existing prefix
    

    delta_table = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_table.alias("t")
        .merge(source_df.alias("s"), "t.plate_prefix = s.plate_prefix")
        .whenMatchedUpdate(set={"Entity": "s.Entity"})
        .whenNotMatchedInsert(values={"plate_prefix": "s.plate_prefix", "Entity": "s.Entity"})
        .execute()
    )
    print(f"Upserted {TARGET_TABLE} from {source_df.count()} source rows.")

display(spark.table(TARGET_TABLE).orderBy("plate_prefix"))
print(f"Target table row count: {spark.table(TARGET_TABLE).count()}")